# Pixel Art por K-means e Redução de Resolução

**Projeto final da disciplina de Computação Científica e Análise de Dados**

## Motivação e Modelagem

Transformar uma imagem em pixel art envolve resolver dois problemas diferentes:

1. **Redução de resolução:** dividir a imagem original em blocos e representar cada bloco como um único "pixel grande" (a média das cores do bloco)
2. **Redução de paleta de cores:** limitar a imagem a apenas $k$ cores, escolhidas de forma que representem bem todas as cores originais

A redução do número de cores é exatamente um problema de **clusterização**, porque cada pixel é um ponto em $\mathbb{R}^3$ (R, G, B), e queremos agrupá-los em $k$ clusters que minimizem a soma dos quadrados das distâncias de cada ponto ao centro do seu cluster. Isso é um problema de **Reconhecimento de Padrões** resolvido por **K-means**, conforme o conteúdo aprendido durante o curso.

Etapas:

1. Setup
2. Redução da resolução (blocos → pixels grandes)
3. K-means na paleta de cores
4. Escolha do k (método do cotovelo)
5. Reconstrução e comparação visual


## Etapa 1 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans

%matplotlib inline
np.random.seed(42)

### Upload de Imagem

A célula abaixo utiliza a biblioteca do google para abrir um seletor de arquivos diretamente no Colab. Ao testar com uma nova imagem, basta rodar a célula e escolher o arquivo.

In [ ]:
from google.colab import files

uploaded = files.upload()                # abre a caixa de seleção de arquivo
nome_arquivo = list(uploaded.keys())[0]  # pega o nome do arquivo escolhido

imagem = Image.open(nome_arquivo).convert('RGB')
img_vetor = np.array(imagem)

plt.figure(figsize=(5, 5))
plt.imshow(imagem)
plt.title(f"{nome_arquivo} — dimensões: {img_vetor.shape}")
plt.axis('off')
plt.show()

## Etapa 2 — Redução de resolução

In [ ]:
def reduz_res(img_vetor, tamanho_bloco):
    # divide a imagem em blocos NxN e substitui cada bloco pela média das cores
    altura, largura, canais = img_vetor.shape

    altura_dividida = (altura // tamanho_bloco) * tamanho_bloco
    largura_dividida = (largura // tamanho_bloco) * tamanho_bloco
    img_cortada = img_vetor[:altura_dividida, :largura_dividida]

    blocos = img_cortada.reshape(
        altura_dividida // tamanho_bloco, tamanho_bloco,
        largura_dividida // tamanho_bloco, tamanho_bloco,
        canais
    )
    medias = blocos.mean(axis=(1, 3))

    resultado = np.repeat(np.repeat(medias, tamanho_bloco, axis=0), tamanho_bloco, axis=1)
    return resultado.astype(np.uint8)


TAM_BLOCO = 30  # ajustar conforme o tamanho da sua imagem
img_reduzida = reduz_res(img_vetor, TAM_BLOCO)

fig, axs = plt.subplots(1, 2, figsize=(9, 4.5))
axs[0].imshow(img_vetor)
axs[0].set_title("Original")
axs[0].axis('off')
axs[1].imshow(img_reduzida, interpolation='nearest')
axs[1].set_title(f"Baixa resolução (bloco={TAM_BLOCO})")
axs[1].axis('off')
plt.show()

## Etapa 3 — Redução de paleta de cores (K-means)

In [ ]:
def quantizar_cores(img_vetor, k, amostra=10000):
    # reduz a imagem para k cores
    # treina primeiro na amostra de pixels e aplica o resultado na imagem toda
    altura, largura, canais = img_vetor.shape
    pixels = img_vetor.reshape(-1, canais).astype(np.float64)

    n_pixels = pixels.shape[0]
    # se a imagem tiver mais de 10.000 pixels
    if n_pixels > amostra:
        i = np.random.choice(n_pixels, amostra, replace=False) # sorteia um indice aleatorio entre os pixels da imagem, sem repeticao
        pixels_treino = pixels[i]
    else:
        pixels_treino = pixels

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels_treino)

    labels_completo = kmeans.predict(pixels)
    paleta = kmeans.cluster_centers_.astype(np.uint8)

    img_quantizada = paleta[labels_completo].reshape(altura, largura, canais)
    return img_quantizada, paleta


K = 6  # número de cores da pixel art. Ajustado apos o metodo do cotovelo na proxima etapa
img_kmeans, paleta = quantizar_cores(img_reduzida, K)

fundo, imagens = plt.subplots(1, 3, figsize=(13, 4.5))
imagens[0].imshow(img_vetor)
imagens[0].set_title("Original")
imagens[0].axis('off')
imagens[1].imshow(img_reduzida, interpolation='nearest')
imagens[1].set_title("Baixa resolução")
imagens[1].axis('off')
imagens[2].imshow(img_kmeans.astype(np.uint8), interpolation='nearest')
imagens[2].set_title(f"Pixel Art (bloco={TAM_BLOCO}, k={K})")
imagens[2].axis('off')
plt.show()

print("Paleta de cores encontrada (RGB):")
print(paleta)

## Etapa 4 — Escolha do k (Método do Cotovelo)

In [ ]:
def calcular_inercias(img_vetor, k_valores, amostra=10000):
    # calcula a inercia (soma dos quadrados das distâncias) para cada k
    pixels = img_vetor.reshape(-1, 3).astype(np.float64)
    n_pixels = pixels.shape[0]
    if n_pixels > amostra:
        idx = np.random.choice(n_pixels, amostra, replace=False)
        pixels_treino = pixels[idx]
    else:
        pixels_treino = pixels

    inercias = []
    for k in k_valores:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(pixels_treino)
        inercias.append(kmeans.inertia_)
    return inercias


k_valores = range(1, 13)
inercias = calcular_inercias(img_reduzida, k_valores)

plt.figure(figsize=(6, 4))
plt.plot(list(k_valores), inercias, marker='o')
plt.xlabel("k")
plt.ylabel("erro quadrático total")
plt.title("Método do Cotovelo")
plt.grid(alpha=0.3)
plt.show()

## Etapa 5 — Comparação visual entre diferentes valores de k

In [ ]:
k_testados = [1, 3, 4, 6, 8, 12]

tela, imagens = plt.subplots(2, 3, figsize=(12, 9))
imagens = imagens.flatten()

for img, k in zip(imagens, k_testados):
    img_k, _ = quantizar_cores(img_reduzida, k=k)
    img.imshow(img_k.astype(np.uint8), interpolation='nearest')
    img.set_title(f"k = {k}")
    img.axis('off')

plt.suptitle(f"Pixel Art — variando k (bloco = {TAM_BLOCO})", fontsize=14)
plt.tight_layout()
plt.show()

## Conclusões

- A redução da resolução controla a resolução espacial da pixel art, pois blocos maiores deixam a imagem mais estilizada, mas apagam detalhes finos
- O K-means encontra a quantidade de cores como um problema de clusterização em ℝ³, minimizando a soma dos quadrados das distâncias de cada pixel ao centro do seu cluster
- O método do cotovelo mostrou que o ganho de reduzir o erro cai fortemente entre k=1 e k=4, e praticamente fica constante depois disso
- O k=6 foi escolhido como valor final, porque preserva melhor detalhes relevantes da imagem que se perderiam com um k menor, mesmo que o ganho de k=4 para k=6 já seja pequeno.
- Isso mostra uma diferença importante entre otimizar um critério matemático (erro quadrático) e otimizar a percepção visual humana. O cotovelo é um guia numérico, mas a escolha final também depende da avaliação da qualidade do resultado.